## 1 setup

In [1]:
%pip install -q "datasets==3.6.0" pandas pyarrow numpy scipy scikit-learn tqdm matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import ast #per convertire liste e diz python in oggetti veri
import json 
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display #per mostrare meglio tabelle e DataFrame
from scipy.sparse import csr_matrix #per matrici sparse
from sklearn.decomposition import TruncatedSVD #per fare decomposizione della matrice utente-prodotto
from sklearn.neighbors import NearestNeighbors #per trovare elementi simili
from tqdm.auto import tqdm

pd.set_option("display.max_columns",80)
pd.set_option("display.max_colwidth",120)

c:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
CATEGORY="AmazonLike_v2"
DATASET_NAME="test_dataset3"
RANDOM_STATE=42
K_VALUES=(5,10,20)
PRIMARY_METRIC="ndcg" 
#dobbiamo usare ndcg come metrica, perchè misura quanto bene il modello ordina gli elementi
#nelle prime k posizioni
#sommatoria per le prime k pos di (rilevanza elemento in posizione i/log2(i+1))
#se prodotto fuori dalla topk è uguale a 0
PRIMARY_K=10
#scegli che modelli seguire
RUN_POPULARITY=True
RUN_ITEMKNN=True
RUN_SVD=True

USE_RATING_VALUES=False #stimiamo se utente interagisce col prodotto, non il rating che deve dare
MAX_EVAL_ROWS=None #quante righe usare al max nella valutazione, con none le usi tutte
SAVE_RANKS=True #salva i rank prodotti dai modelli
SAVE_MODEL_ARTIFACTS=True #salva artefatti,mapping..

ITEMKNN_NEIGHBORS_LIST=[5,10,50,100] #guarda 50 prodotti vicini/simili
ITEMKNN_N_JOBS=-1 #usa tutte le risorse disponibili

SVD_FACTORS=[2,4,6,8,10,12,16,24,32] #fattori latenti da provare per truncatedSVD per approssimare la matrice utenti-prodotti
SVD_N_ITER=7 #num iterazioni di svd

DEMO_USERS_N=3 #3 utenti demo
DEMO_TOP_N=10 #mostra 10 raccomandazioni per utente

## 2 percorsi e caricamento dati

In [4]:
PROJECT_ROOT=Path(r"C:\Users\arsid\Desktop\Tesi-ecommerce")
RECOMMENDER_DIR=PROJECT_ROOT/"data"/"processed"/f"{DATASET_NAME}_recommender"
NOTEBOOK04_DIR=RECOMMENDER_DIR/"notebook04_models"
NOTEBOOK04_DIR.mkdir(parents=True,exist_ok=True)

TRAIN_PATH=RECOMMENDER_DIR/"train.parquet"
VALIDATION_EVAL_PATH=RECOMMENDER_DIR/"validation_eval.parquet"
TEST_EVAL_PATH=RECOMMENDER_DIR/"test_eval.parquet"
VALIDATION_NEGATIVE_PATH=RECOMMENDER_DIR/"negative_samples_validation.parquet"
TEST_NEGATIVE_PATH=RECOMMENDER_DIR/"negative_samples_test.parquet"
ITEMS_PATH=RECOMMENDER_DIR/"items.parquet"
USER_MAPPING_PATH=RECOMMENDER_DIR/"user_mapping.parquet"
ITEM_MAPPING_PATH=RECOMMENDER_DIR/"item_mapping.parquet"
SPLIT_SUMMARY_PATH=RECOMMENDER_DIR/"split_summary.json"

print("Project root:",PROJECT_ROOT)
print("Recommender dir:",RECOMMENDER_DIR)
print("Notebook04 dir:",NOTEBOOK04_DIR)

Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Recommender dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset3_recommender
Notebook04 dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset3_recommender\notebook04_models


In [5]:
train_df=pd.read_parquet(TRAIN_PATH)
validation_eval_df=pd.read_parquet(VALIDATION_EVAL_PATH)
test_eval_df=pd.read_parquet(TEST_EVAL_PATH)
validation_samples_df=pd.read_parquet(VALIDATION_NEGATIVE_PATH)
test_samples_df=pd.read_parquet(TEST_NEGATIVE_PATH)
items_df=pd.read_parquet(ITEMS_PATH)
user_mapping_df=pd.read_parquet(USER_MAPPING_PATH)
item_mapping_df=pd.read_parquet(ITEM_MAPPING_PATH)
split_summary=json.loads(SPLIT_SUMMARY_PATH.read_text(encoding="utf-8"))

loaded_shapes=pd.DataFrame([
    {"name":"train","rows":len(train_df),"cols":train_df.shape[1]},
    {"name":"validation_eval","rows":len(validation_eval_df),"cols":validation_eval_df.shape[1]},
    {"name":"test_eval","rows":len(test_eval_df),"cols":test_eval_df.shape[1]},
    {"name":"validation_samples","rows":len(validation_samples_df),"cols":validation_samples_df.shape[1]},
    {"name":"test_samples","rows":len(test_samples_df),"cols":test_samples_df.shape[1]},
    {"name":"items","rows":len(items_df),"cols":items_df.shape[1]},
    {"name":"user_mapping","rows":len(user_mapping_df),"cols":user_mapping_df.shape[1]},
    {"name":"item_mapping","rows":len(item_mapping_df),"cols":item_mapping_df.shape[1]},
])

display(loaded_shapes)
display(pd.DataFrame.from_dict(split_summary,orient="index",columns=["value"]).head(30))

,name,rows,cols
0,train,517,19
1,validation_eval,332,19
2,test_eval,332,19
3,validation_samples,332,8
4,test_samples,332,8
5,items,72,17
6,user_mapping,332,2
7,item_mapping,72,2


,value
category,AmazonLike_v2
dataset_name,test_dataset3
source_config,user3_item5
positive_rating_threshold,4
min_user_interactions_after,3
min_item_interactions_after,8
n_negatives,50
negative_sampling_mode,mixed_category
same_category_negative_ratio,0.5
random_state,42


In [6]:
def parse_negative_items(value): #prende valore della colonna negative_item_idxs e prova a convertirlo in lista
    if value is None:
        return []
    if isinstance(value,np.ndarray): #se array lo converte in lista python
        value=value.tolist()
    if isinstance(value,(list,tuple,set)): #se già lista, converte ogni elemento in int
        return [int(item) for item in value]
    if isinstance(value,str): #se valore è una stringa, toglie spazi inutili e prova a interpretarla come lista python
        value=value.strip()
        if not value:
            return []
        parsed=ast.literal_eval(value)
        return [int(item) for item in parsed]
    try:
        if pd.isna(value):
            return []
    except (TypeError,ValueError):
        pass
    return [int(value)]

validation_samples_df=validation_samples_df.copy()
test_samples_df=test_samples_df.copy()

validation_samples_df["negative_item_idxs"]=validation_samples_df["negative_item_idxs"].apply(parse_negative_items)
test_samples_df["negative_item_idxs"]=test_samples_df["negative_item_idxs"].apply(parse_negative_items)

if MAX_EVAL_ROWS is not None: #se non nullo, limita righe usate per valutazione
    validation_samples_df=validation_samples_df.head(int(MAX_EVAL_ROWS)).copy()
    test_samples_df=test_samples_df.head(int(MAX_EVAL_ROWS)).copy()

print("Validation samples:",len(validation_samples_df))
print("Test samples:",len(test_samples_df))
display(validation_samples_df.head())

Validation samples: 332
Test samples: 332


,split,user_idx,user_id,positive_item_idx,positive_parent_asin,positive_source_category,negative_item_idxs,num_negatives
0,validation,0,AE2CTC3YNHGSDFX4IMHYHTOPAGOQ,22,B003NSLGW2,Video Games,"[65, 11, 19, 17, 37, 71, 33, 4, 39, 6, 48, 56, 45, 1, 20, 52, 50, 60, 59, 2, 61, 3, 38, 64, 67, 63, 70, 16, 23, 68, ...",50
1,validation,1,AE3MR2EVAQWHB7PZCHM2NMMVQBAQ,20,B002Z01QO2,Video Games,"[16, 38, 69, 58, 71, 8, 29, 26, 61, 57, 3, 12, 15, 43, 14, 37, 24, 50, 60, 19, 66, 36, 23, 28, 7, 46, 11, 67, 2, 52,...",50
2,validation,2,AE3XC6CB6EMB7TPIP3IM5DH3WCKA,6,B001GIUWX0,Video Games,"[43, 70, 5, 56, 9, 52, 17, 66, 40, 0, 46, 15, 11, 50, 71, 26, 53, 45, 51, 24, 62, 23, 58, 18, 41, 12, 10, 34, 63, 28...",50
3,validation,3,AE4SA5LYAQUYW3JKK4KSJZAXE66A,51,B00CMQTUSS,Video Games,"[46, 47, 64, 28, 13, 58, 27, 71, 16, 68, 2, 1, 63, 50, 65, 0, 69, 6, 52, 57, 48, 41, 35, 54, 4, 10, 22, 44, 32, 70, ...",50
4,validation,4,AE5LTT322JJKPDGGF5SITHSSIBFA,57,B00DC7G0GG,Video Games,"[25, 29, 34, 52, 13, 62, 56, 69, 15, 8, 5, 20, 2, 16, 47, 26, 30, 7, 70, 35, 41, 10, 19, 24, 38, 50, 59, 11, 22, 1, ...",50


## 3 matrice utente prodotto

In [7]:
#calcoliamo numero totale di utenti e prodotti
n_users=int(max(user_mapping_df["user_idx"].max(),train_df["user_idx"].max())+1) 
n_items=int(max(item_mapping_df["item_idx"].max(),train_df["item_idx"].max())+1)
#estraiamo dal training set due array numpy
train_user_idx=train_df["user_idx"].astype(np.int64).to_numpy()
train_item_idx=train_df["item_idx"].astype(np.int64).to_numpy()

if USE_RATING_VALUES and "rating" in train_df.columns: #se true, allora prendi rating dal dataset
    train_values=train_df["rating"].astype(np.float32).to_numpy()
else:
    train_values=np.ones(len(train_df),dtype=np.float32) #mette 1 se esiste l'interazione, non ci interessa il rating

#costruisci matrice sparsa, righe=utenti, colonne=prodotti, valori=interazioni
train_matrix=csr_matrix((train_values,(train_user_idx,train_item_idx)),shape=(n_users,n_items),dtype=np.float32)
train_matrix.sum_duplicates() #se ci sono duplicati, li somma
train_matrix.eliminate_zeros() #pulisce zero salvati

density=train_matrix.nnz/(n_users*n_items) #calcola densità matrice, nnz=numero celle non nulle

matrix_stats={
    "n_users":int(n_users),
    "n_items":int(n_items),
    "train_interactions":int(train_matrix.nnz),
    "density_pct":float(density*100),
}

display(pd.DataFrame.from_dict(matrix_stats,orient="index",columns=["value"]))

,value
n_users,332.000000
n_items,72.000000
train_interactions,517.000000
density_pct,2.162818


In [8]:
#crea dizionario, per ogni utente crea l'insieme dei prodotti visti nel train
train_seen_by_user=(
    train_df.groupby("user_idx")["item_idx"]
    .apply(lambda values:set(values.astype(int)))
    .to_dict()
)
#unisce in un unico dataframe le interazioni di train,val,test e crea una tabella con tutte le int note dell'utente
all_interactions_for_seen=pd.concat([train_df,validation_eval_df,test_eval_df],ignore_index=True)
#crea altro dizionario, per ogni utente, insieme di tutti i prodotti visti in train+val+test
all_seen_by_user=(
    all_interactions_for_seen.groupby("user_idx")["item_idx"]
    .apply(lambda values:set(values.astype(int)))
    .to_dict()
)
#crea array con tutti prodotti presenti nel training set
train_item_candidates=np.array(sorted(train_df["item_idx"].astype(int).unique()),dtype=np.int64)

print("Utenti con storico train:",len(train_seen_by_user))
print("Item candidati dal train:",len(train_item_candidates))

Utenti con storico train: 332
Item candidati dal train: 72


## 4 funzioni di valutazione

In [9]:
#calcola in che posizione viene messo il prodotto positivo rispetto ai negativi campionati
def rank_positive(user_idx,positive_item_idx,negative_item_idxs,score_fn):
    #costruisce lista dei prodotti candidati da ordinare
    candidate_items=[int(positive_item_idx)]+[int(item) for item in negative_item_idxs]
    #modello assegna punteggio a ogni prodotto candidato
    scores=score_fn(int(user_idx),candidate_items)
    scores=np.asarray(scores,dtype=np.float64) #converte punteggi in array numpy
    scores=np.nan_to_num(scores,nan=-np.inf,posinf=np.inf,neginf=-np.inf)
    positive_score=float(scores[0]) #punteggio del nostro prodotto positivo
    rank=1
    for item_idx,score in zip(candidate_items[1:],scores[1:]): #scorre solo i prodotti negativi , con [1:] salta il primo
        if score>positive_score: #se prod neg ha punteggio piu alto, peggiora di uno il rank del positivo
            rank+=1
        elif score==positive_score and int(item_idx)<int(positive_item_idx): #vince item con id piu piccolo
            rank+=1
    return int(rank),positive_score #rank=posizione del prodotto tra i negativi+positivo, positive_score=punteggio assegnato

In [10]:
#prende una lista dei rank ottenuti, e i valori di k da valutare
def calculate_topk_metrics(ranks,k_values):
    ranks_array=np.asarray(ranks,dtype=np.float64) #converte lista in array numpy
    rows=[]
    for k in k_values: #scorre tutti i k
        hits=ranks_array<=int(k) #calcola per ogni esempio se il prodotto positivo è entrato in topk
        hitrate=float(np.mean(hits)) if len(ranks_array) else np.nan #calcola % casi in cui il positivo è in topk
        ndcg_values=np.where(hits,1.0/np.log2(ranks_array+1.0),0.0) #calcola valore ndcg@k per ogni esempio
        #where- se hits true calcola la funzione, se è false mette 0
        mrr_values=np.where(hits,1.0/ranks_array,0.0) #calcola mrr#k cioè mean reciprocal rank
        #misura quanto in alto il modello posiziona il prodotto positivo reale, 1/rank
        rows.append({
            "K":int(k),
            "hitrate":hitrate,
            "recall":hitrate,
            "ndcg":float(np.mean(ndcg_values)) if len(ranks_array) else np.nan,
            "mrr":float(np.mean(mrr_values)) if len(ranks_array) else np.nan,
            "n_examples":int(len(ranks_array)),
        })
    return pd.DataFrame(rows)

In [11]:
#valuta un modello di raccomandazione, su val o test
def evaluate_model(samples_df,model_name,score_fn,split_name,k_values=K_VALUES):
    #sampledf df con pos e neg campionati
    ranks=[] #contiene solo rank dei prodotti positivi
    rank_rows=[] #contiene info per ogni esempio
    iterator=samples_df.itertuples(index=False) #trasforma df in iteratore riga per riga
    for row in tqdm(iterator,total=len(samples_df),desc=f"Evaluate {model_name} {split_name}"): #scorre righe in samplesdf
        #restituisce rank la pos nel ranking, e positive_score il punteggio dato
        rank,positive_score=rank_positive(row.user_idx,row.positive_item_idx,row.negative_item_idxs,score_fn)
        positive_source_category=getattr(row,"positive_source_category",None) #categoria
        ranks.append(rank)
        rank_rows.append({
            "model":model_name,
            "split":split_name,
            "user_idx":int(row.user_idx),
            "positive_item_idx":int(row.positive_item_idx),
            "positive_rank":int(rank),
            "positive_score":float(positive_score),
            "num_negatives":int(len(row.negative_item_idxs)),
            "positive_source_category":positive_source_category,
        })
    metrics_df=calculate_topk_metrics(ranks,k_values) #calcola le metriche topk partendo dai rank
    metrics_df.insert(0,"split",split_name)
    metrics_df.insert(0,"model",model_name)
    ranks_df=pd.DataFrame(rank_rows)
    return metrics_df,ranks_df

## 5 baseline most popular

In [12]:
#costruiamo e valutiamo la baseline

popularity_series=train_df["item_idx"].value_counts() #conta quante volte un prodotto compare nel training set
popularity_scores={int(item_idx):float(count) for item_idx,count in popularity_series.items()} #trasforma la series in diz python

def score_popularity(user_idx,item_idxs): #funzione di scoring, input utente da valutare, e lista di prodotti candidati
    return np.array([popularity_scores.get(int(item_idx),0.0) for item_idx in item_idxs],dtype=np.float64)
    #output un array con un punteggio per ogni prodotto candidato

model_score_functions={} #salva funzioni di scoring dei modelli
validation_metric_frames=[] #salva metriche aggregate dei modelli su val
validation_rank_frames=[] #salva i rank dettagliati dei modelli su val

if RUN_POPULARITY: 
    model_score_functions["MostPopular"]=score_popularity
    #valuta mostpopular sul validation set
    popularity_validation_metrics,popularity_validation_ranks=evaluate_model(validation_samples_df,"MostPopular",score_popularity,"validation")
    
    validation_metric_frames.append(popularity_validation_metrics) #aggiunge i risultati alle liste generali
    validation_rank_frames.append(popularity_validation_ranks)
    display(popularity_validation_metrics)

Evaluate MostPopular validation: 100%|██████████| 332/332 [00:00<00:00, 47865.70it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,MostPopular,validation,5,0.265060,0.265060,0.177870,0.149247,332
1,MostPopular,validation,10,0.343373,0.343373,0.202763,0.159280,332
2,MostPopular,validation,20,0.539157,0.539157,0.251183,0.172012,332


## 6 itemknn
similarità fra prodotti a partire dagli utenti che li hanno valutati positivamente
per un utente un prodotto riceve punteggio alto se è simile a prodotti già presenti nel suo storico di training

In [13]:
itemknn_artifacts={}
itemknn_neighbor_indices=None #indici dei vicini
itemknn_neighbor_scores=None #punteggi di similarità
empty_int_array=np.array([],dtype=np.int64)
empty_float_array=np.array([],dtype=np.float32)

if RUN_ITEMKNN and n_items>1: 
    itemknn_neighbors_list=sorted(set(int(k) for k in ITEMKNN_NEIGHBORS_LIST if int(k)>0)) #pulisce lista dei valori di k
    #converte ogni k in intero, rimuove valori duplicati e ordina i valori crescenti
    if itemknn_neighbors_list:
        adjusted_train_matrix=train_matrix.tocsr(copy=True).astype(np.float32) #copia matrice utenti x prodotti per adjusted cosine
        user_interaction_counts=np.diff(adjusted_train_matrix.indptr) #numero interazioni osservate per utente
        user_rating_sums=np.asarray(adjusted_train_matrix.sum(axis=1)).ravel().astype(np.float32) #somma valori osservati per utente
        user_means=np.divide(user_rating_sums,user_interaction_counts,out=np.zeros_like(user_rating_sums,dtype=np.float32),where=user_interaction_counts>0) #media utente
        for user_idx in range(n_users):
            row_start,row_end=adjusted_train_matrix.indptr[user_idx],adjusted_train_matrix.indptr[user_idx+1]
            if row_end>row_start:
                adjusted_train_matrix.data[row_start:row_end]-=user_means[user_idx] #centra rating/interazioni dell'utente
        adjusted_train_matrix.eliminate_zeros() #rimuove eventuali valori azzerati dal centering
        item_user_matrix=adjusted_train_matrix.T.tocsr() #prodotti x utenti centrati per adjusted cosine
        max_itemknn_neighbors=int(min(max(itemknn_neighbors_list)+1,n_items)) #prende valore massimo della lista
        itemknn_model=NearestNeighbors(n_neighbors=max_itemknn_neighbors,metric="cosine",algorithm="brute",n_jobs=ITEMKNN_N_JOBS) #cosine sui vettori centrati = adjusted cosine
        #adjusted cosine: sottrae la media dell'utente dai valori osservati, poi confronta prodotti con coseno
        itemknn_model.fit(item_user_matrix) #prepara modello sulla matrice
        distances,indices=itemknn_model.kneighbors(item_user_matrix,return_distance=True) #cerca i vicini per ogni prodotto
        similarities=(1.0-distances).astype(np.float32) #trasforma distanze in similarità
        itemknn_neighbor_indices=indices.astype(np.int64) 
        itemknn_neighbor_scores=similarities

        for k in itemknn_neighbors_list: #cicla sui vari k
            model_name=f"ItemKNN_{int(k)}" 
            target_neighbors=int(min(int(k),n_items-1)) #decide quanti vicini usare per quel modello
            current_lookup={}
            for item_idx in range(n_items): #cicla tutti i prodotti, per ogni prodotto costruisce lista dei suoi vicini validi
                current_indices=itemknn_neighbor_indices[item_idx] #prende vicini 
                current_scores=itemknn_neighbor_scores[item_idx] #e suoi punteggi
                keep_mask=(current_indices!=item_idx)&(current_scores>0) #rimuove prodotto stesso e con similarità = 0
                filtered_indices=current_indices[keep_mask][:target_neighbors]
                filtered_scores=current_scores[keep_mask][:target_neighbors]
                current_lookup[int(item_idx)]=(filtered_indices,filtered_scores)
            itemknn_artifacts[model_name]={"k":target_neighbors,"neighbor_lookup":current_lookup}

        print("ItemKNN costruito per K:",itemknn_neighbors_list)
        print("Matrice vicini massima:",itemknn_neighbor_indices.shape)
    else:
        print("ItemKNN saltato: lista vicini vuota")
else:
    print("ItemKNN saltato")

ItemKNN costruito per K: [5, 10, 50, 100]
Matrice vicini massima: (72, 72)


In [14]:
def make_itemknn_score_fn(current_neighbor_lookup): #funzione che riceve una configurazione di vicini
    def score_itemknn(user_idx,item_idxs):  #calcola punteggi itemknn
        #input utente da valutare e prodotti candidati
        history=train_seen_by_user.get(int(user_idx),set())  #recupera prodotti visti dall'utente nel training
        scores=np.zeros(len(item_idxs),dtype=np.float64)
        if not history:
            return scores
        for pos,item_idx in enumerate(item_idxs): #scorre prodotti candidati
            neigh_items,neigh_scores=current_neighbor_lookup.get(int(item_idx),(empty_int_array,empty_float_array))
            #recupera vicini del prodotto candidato
            total_score=0.0
            for neigh_item,neigh_score in zip(neigh_items,neigh_scores): #scorre insieme prodotti vicini e loro punteggi di similarità
                if int(neigh_item) in history: #se un prodotto vicino è stato visto, aggiunge similarità al punteggio
                    total_score+=float(neigh_score)
            scores[pos]=total_score
        return scores
    return score_itemknn

if RUN_ITEMKNN and itemknn_artifacts:
    for model_name,artifact in itemknn_artifacts.items():
        score_fn=make_itemknn_score_fn(artifact["neighbor_lookup"]) #funzione di scoring
        artifact["score_fn"]=score_fn
        model_score_functions[model_name]=score_fn 
        itemknn_validation_metrics,itemknn_validation_ranks=evaluate_model(validation_samples_df,model_name,score_fn,"validation")
        #valuta questa versione sul validation set
        validation_metric_frames.append(itemknn_validation_metrics)
        validation_rank_frames.append(itemknn_validation_ranks)
        print(model_name,"neighbors",artifact["k"])
        display(itemknn_validation_metrics)
else:
    print("ItemKNN validation saltata")

Evaluate ItemKNN_5 validation: 100%|██████████| 332/332 [00:00<00:00, 11464.94it/s]

ItemKNN_5 neighbors 5


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,ItemKNN_5,validation,5,0.225904,0.225904,0.156586,0.133735,332
1,ItemKNN_5,validation,10,0.292169,0.292169,0.177422,0.141992,332
2,ItemKNN_5,validation,20,0.457831,0.457831,0.219040,0.153282,332


Evaluate ItemKNN_10 validation: 100%|██████████| 332/332 [00:00<00:00, 12596.42it/s]

ItemKNN_10 neighbors 10


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,ItemKNN_10,validation,5,0.240964,0.240964,0.167720,0.143725,332
1,ItemKNN_10,validation,10,0.319277,0.319277,0.193049,0.154179,332
2,ItemKNN_10,validation,20,0.487952,0.487952,0.234987,0.165336,332


... (2 outputs left out)

,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,ItemKNN_50,validation,5,0.246988,0.246988,0.169806,0.144478,332
1,ItemKNN_50,validation,10,0.328313,0.328313,0.195440,0.154663,332
2,ItemKNN_50,validation,20,0.512048,0.512048,0.241543,0.167146,332


Evaluate ItemKNN_100 validation: 100%|██████████| 332/332 [00:00<00:00, 10423.67it/s]

ItemKNN_100 neighbors 71


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,ItemKNN_100,validation,5,0.246988,0.246988,0.169806,0.144478,332
1,ItemKNN_100,validation,10,0.328313,0.328313,0.195440,0.154663,332
2,ItemKNN_100,validation,20,0.512048,0.512048,0.241543,0.167146,332


## 7 matrix factorization con truncated svd
Truncated SVD apprende rappresentazioni latenti di utenti e prodotti a partire dalla matrice di training
Il punteggio utente-prodotto è il prodotto scalare tra embedding utente ed embedding prodotto

In [15]:
svd_artifacts={} #dizionario per salvare modelli SVD addestrati con diversi numeri di componenti
#funzione che da scoring SVD
def make_svd_score_fn(user_embeddings,item_embeddings): #input vettori latenti degli utenti, vettori latenti dei prodotti
    def score_svd(user_idx,item_idxs): #funzione interna che assegna punteggio ai prodotti latenti per un certo utente
        user_vector=user_embeddings[int(user_idx)] #prende vettore latente dell'utente
        item_matrix=item_embeddings[np.asarray(item_idxs,dtype=np.int64)] 
        #prende vettori latenti dei prodotti candidati, converte lista dei prodotti in array di interi
        #per poter indicizzare item_embeddings
        return item_matrix@user_vector #calcola prodotto scalare tra ogni prodotto candidato e l'utente
    return score_svd #restituisce la funzione

def train_svd_model(n_components): #funzione che addestra modello svd
    max_components=max(1,min(n_users,n_items)-1) #max n di componenti
    actual_components=int(min(int(n_components),max_components)) #quante componenti usare davvero
    svd=TruncatedSVD(n_components=actual_components,n_iter=SVD_N_ITER,random_state=RANDOM_STATE) #crea il modello TruncatedSVD
    #ncomponent = n dim latenti, niter = numero iterazioni
    user_embeddings=svd.fit_transform(train_matrix).astype(np.float32) #addestra svd sulla matrice di training e produce gli embedding degli utenti
    #user embeddings ha forma utenti x componenti
    item_embeddings=svd.components_.T.astype(np.float32) #estrae embeddings dei prodotti
    #matrice del tipo prodotti x componenti 
    score_fn=make_svd_score_fn(user_embeddings,item_embeddings) #funzione di scoring per questo specifico modello SVD
    artifact={
        "svd":svd,
        "user_embeddings":user_embeddings,
        "item_embeddings":item_embeddings,
        "explained_variance_ratio":float(svd.explained_variance_ratio_.sum()),
        "requested_components":int(n_components),
        "actual_components":int(actual_components),
        "score_fn":score_fn,
    }
    return artifact

In [16]:
if RUN_SVD and n_users>1 and n_items>1:
    for n_factors in SVD_FACTORS: #scorre tutti i valori latenti che vogliamo provare
        model_name=f"SVD_{int(n_factors)}"
        artifact=train_svd_model(n_factors) #addestra e restituisce dizionario con i vari dati
        svd_artifacts[model_name]=artifact
        model_score_functions[model_name]=artifact["score_fn"] #salva funzione di scoring
        svd_validation_metrics,svd_validation_ranks=evaluate_model(validation_samples_df,model_name,artifact["score_fn"],"validation") 
        #valuta modello svd sul validation set
        validation_metric_frames.append(svd_validation_metrics) 
        validation_rank_frames.append(svd_validation_ranks)
        print(model_name,"explained_variance_ratio",artifact["explained_variance_ratio"])
        display(svd_validation_metrics)
else:
    print("SVD saltato")

Evaluate SVD_2 validation: 100%|██████████| 332/332 [00:00<00:00, 46808.60it/s]

SVD_2 explained_variance_ratio 0.08211833238601685


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_2,validation,5,0.240964,0.240964,0.168771,0.145080,332
1,SVD_2,validation,10,0.349398,0.349398,0.203280,0.158993,332
2,SVD_2,validation,20,0.521084,0.521084,0.245743,0.170147,332


Evaluate SVD_4 validation: 100%|██████████| 332/332 [00:00<00:00, 44337.53it/s]

SVD_4 explained_variance_ratio 0.16438829898834229


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_4,validation,5,0.262048,0.262048,0.177297,0.149849,332
1,SVD_4,validation,10,0.373494,0.373494,0.212197,0.163592,332
2,SVD_4,validation,20,0.539157,0.539157,0.253741,0.174827,332


... (17 outputs left out)

,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_24,validation,5,0.186747,0.186747,0.119565,0.097691,332
1,SVD_24,validation,10,0.304217,0.304217,0.156743,0.112566,332
2,SVD_24,validation,20,0.500000,0.500000,0.205567,0.125617,332


Evaluate SVD_32 validation: 100%|██████████| 332/332 [00:00<00:00, 37268.73it/s]

SVD_32 explained_variance_ratio 0.7270126938819885


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_32,validation,5,0.168675,0.168675,0.100010,0.077610,332
1,SVD_32,validation,10,0.262048,0.262048,0.130361,0.090219,332
2,SVD_32,validation,20,0.415663,0.415663,0.168580,0.100402,332


## 8 scelta modello su validation

In [17]:
if not validation_metric_frames:
    raise RuntimeError("Nessun modello valutato su validation")

validation_metrics_df=pd.concat(validation_metric_frames,ignore_index=True) #unisce
validation_ranks_df=pd.concat(validation_rank_frames,ignore_index=True) if validation_rank_frames else pd.DataFrame()

target_validation_df=validation_metrics_df[validation_metrics_df["K"]==PRIMARY_K].copy() #tiene solo il k scelto come valore principale
target_validation_df=target_validation_df.sort_values(PRIMARY_METRIC,ascending=False).reset_index(drop=True) #ordina dal migliore al peggiore

best_model_name=str(target_validation_df.loc[0,"model"]) 
best_validation_score=float(target_validation_df.loc[0,PRIMARY_METRIC])

print("Best model:",best_model_name)
print("Best validation score:",best_validation_score)
display(validation_metrics_df)
display(target_validation_df)

Best model: SVD_4
Best validation score: 0.21219701378846761


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,MostPopular,validation,5,0.265060,0.265060,0.177870,0.149247,332
1,MostPopular,validation,10,0.343373,0.343373,0.202763,0.159280,332
2,MostPopular,validation,20,0.539157,0.539157,0.251183,0.172012,332
3,ItemKNN_5,validation,5,0.225904,0.225904,0.156586,0.133735,332
4,ItemKNN_5,validation,10,0.292169,0.292169,0.177422,0.141992,332
5,ItemKNN_5,validation,20,0.457831,0.457831,0.219040,0.153282,332
6,ItemKNN_10,validation,5,0.240964,0.240964,0.167720,0.143725,332
7,ItemKNN_10,validation,10,0.319277,0.319277,0.193049,0.154179,332
8,ItemKNN_10,validation,20,0.487952,0.487952,0.234987,0.165336,332
9,ItemKNN_50,validation,5,0.246988,0.246988,0.169806,0.144478,332


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_4,validation,10,0.373494,0.373494,0.212197,0.163592,332
1,SVD_2,validation,10,0.349398,0.349398,0.203280,0.158993,332
2,MostPopular,validation,10,0.343373,0.343373,0.202763,0.159280,332
3,SVD_10,validation,10,0.349398,0.349398,0.198879,0.154073,332
4,SVD_6,validation,10,0.355422,0.355422,0.195918,0.148004,332
5,ItemKNN_100,validation,10,0.328313,0.328313,0.195440,0.154663,332
6,ItemKNN_50,validation,10,0.328313,0.328313,0.195440,0.154663,332
7,SVD_8,validation,10,0.349398,0.349398,0.194702,0.148780,332
8,ItemKNN_10,validation,10,0.319277,0.319277,0.193049,0.154179,332
9,SVD_12,validation,10,0.322289,0.322289,0.183977,0.141850,332


## 9 valutazione su test

In [18]:
test_metric_frames=[]
test_rank_frames=[]

for model_name,score_fn in model_score_functions.items():
    test_metrics,test_ranks=evaluate_model(test_samples_df,model_name,score_fn,"test")
    test_metric_frames.append(test_metrics)
    test_rank_frames.append(test_ranks)
    display(test_metrics)

test_metrics_df=pd.concat(test_metric_frames,ignore_index=True) if test_metric_frames else pd.DataFrame()
test_ranks_df=pd.concat(test_rank_frames,ignore_index=True) if test_rank_frames else pd.DataFrame()

Evaluate MostPopular test: 100%|██████████| 332/332 [00:00<00:00, 56762.96it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,MostPopular,test,5,0.316265,0.316265,0.220845,0.189307,332
1,MostPopular,test,10,0.397590,0.397590,0.246143,0.199169,332
2,MostPopular,test,20,0.581325,0.581325,0.291699,0.211201,332


Evaluate ItemKNN_5 test: 100%|██████████| 332/332 [00:00<00:00, 17358.19it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,ItemKNN_5,test,5,0.198795,0.198795,0.154111,0.139207,332
1,ItemKNN_5,test,10,0.250000,0.250000,0.169775,0.145151,332
2,ItemKNN_5,test,20,0.394578,0.394578,0.205188,0.154279,332


Evaluate ItemKNN_10 test: 100%|██████████| 332/332 [00:00<00:00, 13756.30it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,ItemKNN_10,test,5,0.213855,0.213855,0.162024,0.144930,332
1,ItemKNN_10,test,10,0.271084,0.271084,0.179842,0.151886,332
2,ItemKNN_10,test,20,0.394578,0.394578,0.210310,0.159857,332


... (18 outputs left out)

Evaluate SVD_24 test: 100%|██████████| 332/332 [00:00<00:00, 44772.33it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_24,test,5,0.168675,0.168675,0.119861,0.103865,332
1,SVD_24,test,10,0.271084,0.271084,0.151995,0.116560,332
2,SVD_24,test,20,0.457831,0.457831,0.198566,0.129000,332


Evaluate SVD_32 test: 100%|██████████| 332/332 [00:00<00:00, 52575.28it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_32,test,5,0.159639,0.159639,0.108022,0.091064,332
1,SVD_32,test,10,0.246988,0.246988,0.135208,0.101664,332
2,SVD_32,test,20,0.436747,0.436747,0.182206,0.114048,332


In [19]:
all_metrics_df=pd.concat([validation_metrics_df,test_metrics_df],ignore_index=True) #unisce in una unica tabella metriche di val e test

baseline_df=( #tab separata per risultati della baseline most popular
    all_metrics_df[all_metrics_df["model"]=="MostPopular"]
    [["split","K","hitrate","recall","ndcg","mrr"]]
    .rename(columns={
        "hitrate":"hitrate_baseline",
        "recall":"recall_baseline",
        "ndcg":"ndcg_baseline",
        "mrr":"mrr_baseline",
    })
)

comparison_df=all_metrics_df.merge(baseline_df,on=["split","K"],how="left") #unisce tab dei modelli con la tab della baseline

for metric in ["hitrate","recall","ndcg","mrr"]:
    comparison_df[f"{metric}_delta_vs_popularity"]=comparison_df[metric]-comparison_df[f"{metric}_baseline"] #per ogni metrica, calcola quanto è sopra o sotto alla baseline

display(comparison_df.sort_values(["split","K","ndcg"],ascending=[True,True,False]))

,model,split,K,hitrate,recall,ndcg,mrr,n_examples,hitrate_baseline,recall_baseline,ndcg_baseline,mrr_baseline,hitrate_delta_vs_popularity,recall_delta_vs_popularity,ndcg_delta_vs_popularity,mrr_delta_vs_popularity
42,MostPopular,test,5,0.316265,0.316265,0.220845,0.189307,332,0.316265,0.316265,0.220845,0.189307,0.000000,0.000000,0.000000,0.000000
57,SVD_2,test,5,0.246988,0.246988,0.178639,0.156376,332,0.316265,0.316265,0.220845,0.189307,-0.069277,-0.069277,-0.042206,-0.032932
60,SVD_4,test,5,0.228916,0.228916,0.172393,0.153916,332,0.316265,0.316265,0.220845,0.189307,-0.087349,-0.087349,-0.048451,-0.035392
51,ItemKNN_50,test,5,0.216867,0.216867,0.162686,0.144880,332,0.316265,0.316265,0.220845,0.189307,-0.099398,-0.099398,-0.058158,-0.044428
54,ItemKNN_100,test,5,0.216867,0.216867,0.162686,0.144880,332,0.316265,0.316265,0.220845,0.189307,-0.099398,-0.099398,-0.058158,-0.044428
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32,SVD_12,validation,20,0.509036,0.509036,0.231715,0.155223,332,0.539157,0.539157,0.251183,0.172012,-0.030120,-0.030120,-0.019468,-0.016790
5,ItemKNN_5,validation,20,0.457831,0.457831,0.219040,0.153282,332,0.539157,0.539157,0.251183,0.172012,-0.081325,-0.081325,-0.032142,-0.018731
35,SVD_16,validation,20,0.493976,0.493976,0.209225,0.131475,332,0.539157,0.539157,0.251183,0.172012,-0.045181,-0.045181,-0.041958,-0.040537
38,SVD_24,validation,20,0.500000,0.500000,0.205567,0.125617,332,0.539157,0.539157,0.251183,0.172012,-0.039157,-0.039157,-0.045616,-0.046395


In [20]:
#metriche per categoria del prodotto positivo
#servono a capire se il modello funziona solo su una categoria dominante o anche sulle categorie piccole
#prende dataset di ranks e calcola metriche topk per ogni categoria
def calculate_group_metrics(ranks_df,group_col="positive_source_category",k_values=K_VALUES):
    if ranks_df.empty or group_col not in ranks_df.columns:
        return pd.DataFrame()
    rows=[]
    for (model,split,group_value),group_df in ranks_df.groupby(["model","split",group_col],dropna=False):
        metrics_df=calculate_topk_metrics(group_df["positive_rank"].to_numpy(),k_values)
        metrics_df.insert(0,group_col,group_value)
        metrics_df.insert(0,"split",split)
        metrics_df.insert(0,"model",model)
        rows.append(metrics_df)
    return pd.concat(rows,ignore_index=True) if rows else pd.DataFrame()

all_ranks_df=pd.concat([validation_ranks_df,test_ranks_df],ignore_index=True) if not validation_ranks_df.empty or not test_ranks_df.empty else pd.DataFrame()
category_metrics_df=calculate_group_metrics(all_ranks_df)

display(category_metrics_df.sort_values(["split","K","positive_source_category","ndcg"],ascending=[True,True,True,False]))


,model,split,positive_source_category,K,hitrate,recall,ndcg,mrr,n_examples
0,ItemKNN_10,test,Electronics,5,0.000000,0.000000,0.000000,0.000000,2
12,ItemKNN_100,test,Electronics,5,0.000000,0.000000,0.000000,0.000000,2
24,ItemKNN_5,test,Electronics,5,0.000000,0.000000,0.000000,0.000000,2
36,ItemKNN_50,test,Electronics,5,0.000000,0.000000,0.000000,0.000000,2
48,MostPopular,test,Electronics,5,0.000000,0.000000,0.000000,0.000000,2
...,...,...,...,...,...,...,...,...,...
83,SVD_12,validation,Video Games,20,0.510638,0.510638,0.233136,0.156486,329
35,ItemKNN_5,validation,Video Games,20,0.458967,0.458967,0.220278,0.154477,329
95,SVD_16,validation,Video Games,20,0.498480,0.498480,0.211133,0.132674,329
119,SVD_24,validation,Video Games,20,0.504559,0.504559,0.207442,0.126763,329


## 10 salvataggio risultati

In [21]:
METRICS_PATH=NOTEBOOK04_DIR/"notebook04_metrics.csv"
COMPARISON_PATH=NOTEBOOK04_DIR/"notebook04_comparison.csv"
VALIDATION_RANKS_PATH=NOTEBOOK04_DIR/"notebook04_validation_ranks.parquet"
TEST_RANKS_PATH=NOTEBOOK04_DIR/"notebook04_test_ranks.parquet"
SUMMARY_PATH=NOTEBOOK04_DIR/"notebook04_summary.json"
CATEGORY_METRICS_PATH=NOTEBOOK04_DIR/"notebook04_category_metrics.csv"
POPULARITY_SCORES_PATH=NOTEBOOK04_DIR/"popularity_scores.parquet"
ITEMKNN_INDICES_PATH=NOTEBOOK04_DIR/"itemknn_neighbor_indices.npy"
ITEMKNN_SCORES_PATH=NOTEBOOK04_DIR/"itemknn_neighbor_scores.npy"
SVD_USER_EMB_PATH=NOTEBOOK04_DIR/"best_svd_user_embeddings.npy"
SVD_ITEM_EMB_PATH=NOTEBOOK04_DIR/"best_svd_item_embeddings.npy"
SVD_INFO_PATH=NOTEBOOK04_DIR/"best_svd_info.json"

popularity_scores_df=pd.DataFrame({
    "item_idx":list(popularity_scores.keys()),
    "popularity_score":list(popularity_scores.values()),
})


In [22]:
all_metrics_df.to_csv(METRICS_PATH,index=False) #salva tutte le metriche
comparison_df.to_csv(COMPARISON_PATH,index=False) #salva tab confronto con baseline
if not category_metrics_df.empty:
    category_metrics_df.to_csv(CATEGORY_METRICS_PATH,index=False)
popularity_scores_df.to_parquet(POPULARITY_SCORES_PATH,index=False) #salva punteggi popolarità

if SAVE_RANKS:
    validation_ranks_df.to_parquet(VALIDATION_RANKS_PATH,index=False)
    test_ranks_df.to_parquet(TEST_RANKS_PATH,index=False)

if SAVE_MODEL_ARTIFACTS and itemknn_neighbor_indices is not None:
    np.save(ITEMKNN_INDICES_PATH,itemknn_neighbor_indices)
    np.save(ITEMKNN_SCORES_PATH,itemknn_neighbor_scores)

best_svd_name=None
if SAVE_MODEL_ARTIFACTS and svd_artifacts:
    svd_candidates=target_validation_df[target_validation_df["model"].isin(svd_artifacts.keys())]
    if not svd_candidates.empty:
        best_svd_name=str(svd_candidates.iloc[0]["model"])
        best_svd_artifact=svd_artifacts[best_svd_name]
        np.save(SVD_USER_EMB_PATH,best_svd_artifact["user_embeddings"])
        np.save(SVD_ITEM_EMB_PATH,best_svd_artifact["item_embeddings"])
        SVD_INFO_PATH.write_text(json.dumps({
            "model":best_svd_name,
            "requested_components":best_svd_artifact["requested_components"],
            "actual_components":best_svd_artifact["actual_components"],
            "explained_variance_ratio":best_svd_artifact["explained_variance_ratio"],
        },indent=2),encoding="utf-8")

summary={
    "category":CATEGORY,
    "dataset_name":DATASET_NAME,
    "primary_metric":PRIMARY_METRIC,
    "primary_k":PRIMARY_K,
    "best_model_validation":best_model_name,
    "best_validation_score":best_validation_score,
    "models_evaluated":list(model_score_functions.keys()),
    "train_rows":int(len(train_df)),
    "validation_samples":int(len(validation_samples_df)),
    "test_samples":int(len(test_samples_df)),
    "n_users":int(n_users),
    "n_items":int(n_items),
    "metrics_path":str(METRICS_PATH),
    "comparison_path":str(COMPARISON_PATH),
    "category_metrics_path":str(CATEGORY_METRICS_PATH) if not category_metrics_df.empty else None,
    "popularity_scores_path":str(POPULARITY_SCORES_PATH),
    "best_svd_name":best_svd_name,
}

SUMMARY_PATH.write_text(json.dumps(summary,indent=2),encoding="utf-8")

display(pd.DataFrame.from_dict(summary,orient="index",columns=["value"]))

,value
category,AmazonLike_v2
dataset_name,test_dataset3
primary_metric,ndcg
primary_k,10
best_model_validation,SVD_4
best_validation_score,0.212197
models_evaluated,"[MostPopular, ItemKNN_5, ItemKNN_10, ItemKNN_50, ItemKNN_100, SVD_2, SVD_4, SVD_6, SVD_8, SVD_10, SVD_12, SVD_16, SV..."
train_rows,517
validation_samples,332
test_samples,332


## 11 controlli qualitativi

In [23]:
def top_items_for_user(user_idx,score_fn,top_n=DEMO_TOP_N):
    #funzione che restituisce i migliori top_n prodotti per un certo utente
    seen_items=all_seen_by_user.get(int(user_idx),set())
    candidate_items=np.array([item for item in train_item_candidates if int(item) not in seen_items],dtype=np.int64)
    #prodotti candidati
    if len(candidate_items)==0:
        return pd.DataFrame()
    scores=score_fn(int(user_idx),candidate_items) #calcola punteggi di ogni prodotto candidato
    scores=np.asarray(scores,dtype=np.float64)
    scores=np.nan_to_num(scores,nan=-np.inf,posinf=np.inf,neginf=-np.inf)
    order=np.lexsort((candidate_items,-scores))
    top_positions=order[:int(top_n)]
    return pd.DataFrame({
        "user_idx":int(user_idx),
        "rank":np.arange(1,len(top_positions)+1,dtype=np.int64),
        "item_idx":candidate_items[top_positions].astype(int),
        "score":scores[top_positions].astype(float),
    })

def attach_item_metadata(recommendations_df): #funzione che aggiunge info sui prodotti alle raccomandazioni
    if recommendations_df.empty:
        return recommendations_df
    items_for_join=items_df.copy()
    if "item_idx" not in items_for_join.columns:
        items_for_join=items_for_join.merge(item_mapping_df,on="parent_asin",how="left")
    return recommendations_df.merge(items_for_join,on="item_idx",how="left")

In [24]:
demo_user_idxs=(
    test_samples_df["user_idx"]
    .drop_duplicates()
    .head(DEMO_USERS_N) #tiene solo i primi utenti, per demo test
    .astype(int)
    .tolist()
)

best_score_fn=model_score_functions[best_model_name] #recupera funzione migliore, per ora baseline
demo_frames=[]

for user_idx in demo_user_idxs:
    demo_frames.append(top_items_for_user(user_idx,best_score_fn,DEMO_TOP_N)) #per ogni utente genera le raccomandazioni topN

demo_recommendations_df=pd.concat(demo_frames,ignore_index=True) if demo_frames else pd.DataFrame()
demo_recommendations_df=attach_item_metadata(demo_recommendations_df) 

DEMO_RECOMMENDATIONS_PATH=NOTEBOOK04_DIR/"demo_recommendations.csv"
demo_recommendations_df.to_csv(DEMO_RECOMMENDATIONS_PATH,index=False)

preferred_cols=["user_idx","rank","item_idx","parent_asin","title","source_category","category","avg_rating","num_ratings","score"]
display_cols=[col for col in preferred_cols if col in demo_recommendations_df.columns]

display(demo_recommendations_df[display_cols].head(30) if display_cols else demo_recommendations_df.head(30))
print("Demo recommendations path:",DEMO_RECOMMENDATIONS_PATH)

,user_idx,rank,item_idx,parent_asin,title,source_category,category,avg_rating,num_ratings,score
0,0,1,28,B0050SX9I2,Sony PS3 Media/Blu-ray Disc Remote Control,Video Games,Video Games,4.2,1678,0.062705
1,0,2,3,B000OYMYZQ,Xbox 360 Black Play and Charge Kit,Video Games,Video Games,4.5,1862,0.054899
2,0,3,19,B002I0K6X6,Playstation Move Navigation Controller,Video Games,Video Games,4.4,1160,0.054485
3,0,4,11,B002BSA298,Kinect Sensor with Kinect Adventures!,Video Games,Video Games,4.3,2278,0.046499
4,0,5,56,B00DBDPOZ4,Xbox One Play and Charge Kit,Video Games,Video Games,4.6,17062,0.046396
5,0,6,42,B009AGXH64,Nintendo Wii U Console - 32GB Black Deluxe Set,Video Games,Video Games,4.3,1598,0.046318
6,0,7,58,B00DC7G2W8,Mario Kart 8 - Nintendo Wii U,Video Games,Video Games,4.5,3796,0.043570
7,0,8,0,B000FQ2D5E,Metal Gear Solid 4: Guns of the Patriots,Video Games,Video Games,4.7,1691,0.036217
8,0,9,63,B00FM5IY0Q,Dead Rising 3,Video Games,Video Games,4.3,772,0.036190
9,0,10,32,B0053BCO00,Mario Kart 7,Video Games,Video Games,4.7,4141,0.035335


Demo recommendations path: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset3_recommender\notebook04_models\demo_recommendations.csv
